<div dir="rtl">

# الدرس 37: شبكة عصبية من الصفر

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### البيانات: حلزون من ثلاث فئات

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.0, 1.0, n_per_class)                 # radius
        t = np.linspace(4 * k, 4 * (k + 1), n_per_class)       # angle
        t += rng.normal(0, noise, n_per_class)
        X.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)

X, y = make_spiral()
idx = np.random.default_rng(1).permutation(len(X))
X_tr, y_tr = X[idx[:240]], y[idx[:240]]
X_val, y_val = X[idx[240:]], y[idx[240:]]
print("train:", X_tr.shape, " validation:", X_val.shape, " per class:", np.bincount(y))

In [ ]:
plt.figure(figsize=(6.4, 4.0))
for k, marker in zip(range(3), ["o", "s", "^"]):
    plt.scatter(X[y == k, 0], X[y == k, 1], s=16, marker=marker, label=f"class {k}")
plt.legend(fontsize=8); plt.axis("equal"); plt.grid(True)
plt.show()

<div dir="rtl">

### الطبقات

</div>

In [ ]:
class Linear:
    def __init__(self, d_in, d_out, rng):
        self.W = rng.normal(0, np.sqrt(2 / d_in), size=(d_in, d_out))
        self.b = np.zeros(d_out)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, delta):
        self.dW = self.X.T @ delta
        self.db = delta.sum(axis=0)
        return delta @ self.W.T

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]

class ReLU:
    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask

    def backward(self, delta):
        return delta * self.mask

    def params(self):
        return []

<div dir="rtl">

### السوفتماكس مع الخسارة وتدرّجها

</div>

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # numerical stability, lesson 13
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def softmax_cross_entropy(Z, y):
    """Z: logits (n, C). y: class indices (n,). Returns loss, probs, dL/dZ."""
    n = len(y)
    P = softmax(Z)
    loss = -np.mean(np.log(P[np.arange(n), y]))
    grad = P.copy()
    grad[np.arange(n), y] -= 1                # P - one_hot(y)
    return loss, P, grad / n

loss, P, grad = softmax_cross_entropy(np.array([[2.0, 1.0, 0.1]]), np.array([0]))
print("probs:", P.round(3), " loss:", round(loss, 3))
print("grad :", grad.round(3))

<div dir="rtl">

### النموذج، وخسارة البداية

</div>

In [ ]:
def make_model(sizes, seed=0):
    rng = np.random.default_rng(seed)
    layers = []
    for d_in, d_out in zip(sizes[:-1], sizes[1:]):
        layers += [Linear(d_in, d_out, rng), ReLU()]
    return layers[:-1]                     # no ReLU after the last layer

def forward(layers, X):
    for layer in layers:
        X = layer.forward(X)
    return X                               # logits

def backward(layers, delta):
    for layer in reversed(layers):
        delta = layer.backward(delta)

model = make_model([2, 64, 3])
loss0 = softmax_cross_entropy(forward(model, X_tr), y_tr)[0]
print(f"initial loss: {loss0:.3f}   ln(3) = {np.log(3):.3f}")

<div dir="rtl">

### حلقة التدريب

</div>

In [ ]:
def accuracy(layers, X, y):
    return np.mean(forward(layers, X).argmax(axis=1) == y)

def losses(layers):
    return (softmax_cross_entropy(forward(layers, X_tr), y_tr)[0],
            softmax_cross_entropy(forward(layers, X_val), y_val)[0])

def train(layers, epochs=200, batch_size=32, lr=0.5, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    history = [losses(layers)]                             # epoch 0: before training
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(X_tr))                 # shuffle every epoch
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            logits = forward(layers, X_tr[batch])
            loss, _, delta = softmax_cross_entropy(logits, y_tr[batch])
            backward(layers, delta)
            for layer in layers:                            # gradient descent step
                for p, g in layer.params():
                    p -= lr * g
        history.append(losses(layers))
        if verbose and epoch % 40 == 0:
            tr_loss, va_loss = history[-1]
            print(f"epoch {epoch:3}: train loss {tr_loss:.3f}, val loss {va_loss:.3f}, "
                  f"val acc {accuracy(layers, X_val, y_val):.3f}")
    return np.array(history)

history = train(model)

In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
axes[0].plot(history[:, 0], label="train"); axes[0].plot(history[:, 1], label="validation")
axes[0].axhline(np.log(3), ls="--", color="gray", lw=1)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].grid(True)
xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 150), np.linspace(-1.1, 1.1, 150))
pred = forward(model, np.column_stack([xx.ravel(), yy.ravel()])).argmax(axis=1).reshape(xx.shape)
axes[1].contourf(xx, yy, pred, levels=[-0.5, 0.5, 1.5, 2.5], cmap=ListedColormap(["C0", "C1", "C2"]), alpha=0.25)
for k, marker in zip(range(3), ["o", "s", "^"]):
    axes[1].scatter(X[y == k, 0], X[y == k, 1], s=12, marker=marker, color=f"C{k}")
axes[1].set_aspect("equal"); axes[1].set_title("decision regions")
plt.tight_layout()
plt.show()

<div dir="rtl">

### هل احتجنا الطبقة المخفية؟

</div>

In [ ]:
linear_model = make_model([2, 3])
train(linear_model, verbose=False)
print(f"no hidden layer : val acc {accuracy(linear_model, X_val, y_val):.3f}")
print(f"one hidden layer: val acc {accuracy(model, X_val, y_val):.3f}")

<div dir="rtl">

## تمرين: شبكة أعمق

1. عدّ بيدك معاملات شبكة: 2، ثم 64، ثم 64، ثم 3.
2. ابنِها بدالة بناء النموذج، ودرّبها بمعدل تعلّم 0.3.
3. قارن دقة التحقق مع شبكة الطبقة الواحدة.

</div>

In [ ]:
deep = make_model([2, 64, 64, 3])
# عدّ المعاملات، ثم درّب، ثم اطبع الدقة